In [3]:
from pymongo import MongoClient
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt

# Connect to MongoDB
client = MongoClient("mongodb+srv://admin:<password>@cluster0.jwzyj.mongodb.net/?ssl=true")
db = client["hotel_guests"]
collection = db["dining_info"]

# Load data into DataFrame
df = pd.DataFrame(list(collection.find()))

# Convert date columns to datetime
df['check_in_date'] = pd.to_datetime(df['check_in_date'])
df['check_out_date'] = pd.to_datetime(df['check_out_date'])
df['order_time'] = pd.to_datetime(df['order_time'])

# Feature engineering
df['check_in_day'] = df['check_in_date'].dt.dayofweek
df['check_out_day'] = df['check_out_date'].dt.dayofweek
df['check_in_month'] = df['check_in_date'].dt.month
df['check_out_month'] = df['check_out_date'].dt.month
df['stay_duration'] = (df['check_out_date'] - df['check_in_date']).dt.days
df['is_weekend'] = (df['order_time'].dt.dayofweek >= 5).astype(int)

# Split dataset
features_df = df[df['order_time'] < '2024-01-01']
train_df = df[(df['order_time'] >= '2024-01-01') & (df['order_time'] <= '2024-12-01')]
test_df = df[df['order_time'] > '2024-12-01']

if test_df.empty:
    print("⚠ WARNING: Test dataset is empty. Adjust your date split range.")

# Aggregations
customer_features = features_df.groupby('customer_id').agg(
    total_orders_per_customer=('transaction_id', 'count'),
    avg_spend_per_customer=('price_for_1', 'mean')
).reset_index()

cuisine_features = features_df.groupby('Preferred Cusine').agg(
    total_orders_per_cuisine=('transaction_id', 'count')
).reset_index()

cuisine_popular_dish = features_df.groupby('Preferred Cusine')['dish'].agg(
    lambda x: x.mode()[0]
).reset_index().rename(columns={'dish': 'popular_dish_for_this_cuisine'})

# Merge to train and test
for data in [train_df, test_df]:
    data.merge(customer_features, on='customer_id', how='left', inplace=True)
    data.merge(cuisine_features, on='Preferred Cusine', how='left', inplace=True)
    data.merge(cuisine_popular_dish, on='Preferred Cusine', how='left', inplace=True)

# Drop unnecessary columns
cols_to_drop = ['_id', 'transaction_id', 'customer_id', 'price_for_1', 'Qty',
                'order_time', 'check_in_date', 'check_out_date']

train_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=cols_to_drop, inplace=True)

# OneHotEncoding
categorical_cols = ['Preferred Cusine', 'popular_dish_for_this_cuisine']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_train = encoder.fit_transform(train_df[categorical_cols])
encoded_train_df = pd.DataFrame(encoded_train, columns=encoder.get_feature_names_out(categorical_cols), index=train_df.index)
train_df = pd.concat([train_df.drop(columns=categorical_cols), encoded_train_df], axis=1)

encoded_test = encoder.transform(test_df[categorical_cols])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder.get_feature_names_out(categorical_cols), index=test_df.index)
test_df = pd.concat([test_df.drop(columns=categorical_cols), encoded_test_df], axis=1)

# Label encode target
train_df.dropna(subset=['dish'], inplace=True)
test_df.dropna(subset=['dish'], inplace=True)

label_encoder = LabelEncoder()
train_df['dish'] = label_encoder.fit_transform(train_df['dish'])
test_df['dish'] = label_encoder.transform(test_df['dish'])

# Split features/target
X_train = train_df.drop(columns=['dish'])
y_train = train_df['dish']
X_test = test_df.drop(columns=['dish'])
y_test = test_df['dish']

# Train XGBoost model
xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",     # PROBABILITY output
    eval_metric="mlogloss",
    learning_rate=0.05,
    max_depth=4,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=110
)

xgb_model.fit(X_train, y_train)

# Predictions & Score
y_pred = xgb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", accuracy)

y_pred_prob = xgb_model.predict_proba(X_test)
logloss = log_loss(y_test, y_pred_prob)
print("Model Log Loss:", logloss)

# Feature importance
xgb.plot_importance(xgb_model, max_num_features=10)
plt.show()


ServerSelectionTimeoutError: SSL handshake failed: cluster0-shard-00-01.jwzyj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1028) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: cluster0-shard-00-00.jwzyj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1028) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: cluster0-shard-00-02.jwzyj.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1028) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69203a4fc692cb691b36d933, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('cluster0-shard-00-00.jwzyj.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: cluster0-shard-00-00.jwzyj

In [4]:
from pymongo import MongoClient
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt
import certifi

# ----------------------------------------------------------
# CONNECT TO MONGODB
# ----------------------------------------------------------

# Replace <password> with your actual password and REMOVE <>
MONGO_URI = "mongodb+srv://admin:<password>@cluster0.jwzyj.mongodb.net/?retryWrites=true&w=majority"

try:
    client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
    client.admin.command('ping')
    print("✅ Connected to MongoDB successfully!")
except Exception as e:
    print("❌ MongoDB Connection Error:", e)
    raise SystemExit()

db = client["hotel_guests"]
collection = db["dining_info"]

# Load Data
df = pd.DataFrame(list(collection.find()))
print("📦 Records Loaded:", len(df))

# ----------------------------------------------------------
# FEATURE PROCESSING
# ----------------------------------------------------------

df['check_in_date'] = pd.to_datetime(df['check_in_date'])
df['check_out_date'] = pd.to_datetime(df['check_out_date'])
df['order_time'] = pd.to_datetime(df['order_time'])

df['check_in_day'] = df['check_in_date'].dt.dayofweek
df['check_out_day'] = df['check_out_date'].dt.dayofweek
df['check_in_month'] = df['check_in_date'].dt.month
df['check_out_month'] = df['check_out_date'].dt.month
df['stay_duration'] = (df['check_out_date'] - df['check_in_date']).dt.days
df['is_weekend'] = (df['order_time'].dt.dayofweek >= 5).astype(int)

# ----------------------------------------------------------
# SPLIT DATASETS
# ----------------------------------------------------------

historical_df = df[df['order_time'] < "2024-01-01"]
train_df = df[(df['order_time'] >= "2024-01-01") & (df['order_time'] <= "2024-12-01")]
test_df = df[df['order_time'] > "2024-12-01"]

if test_df.empty:
    print("⚠ WARNING: Test set is empty. Adjust date range.")

# ----------------------------------------------------------
# AGGREGATED FEATURES
# ----------------------------------------------------------

customer_features = historical_df.groupby("customer_id").agg(
    total_orders_per_customer=("transaction_id", "count"),
    avg_spend_per_customer=("price_for_1", "mean")
).reset_index()

cuisine_features = historical_df.groupby("Preferred Cusine").agg(
    total_orders_per_cuisine=("transaction_id", "count")
).reset_index()

cuisine_popular_dish = (
    historical_df.groupby("Preferred Cusine")["dish"]
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(columns={"dish": "popular_dish_for_this_cuisine"})
)

# ----------------------------------------------------------
# MERGE FEATURES
# ----------------------------------------------------------

for data in [train_df, test_df]:
    data.merge(customer_features, on="customer_id", how="left", inplace=True)
    data.merge(cuisine_features, on="Preferred Cusine", how="left", inplace=True)
    data.merge(cuisine_popular_dish, on="Preferred Cusine", how="left", inplace=True)

# Drop extra fields
cols_to_drop = ['_id', 'transaction_id', 'customer_id', 'price_for_1', 'Qty',
                'order_time', 'check_in_date', 'check_out_date']

train_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=cols_to_drop, inplace=True)

# ----------------------------------------------------------
# ENCODING
# ----------------------------------------------------------

categorical_cols = ["Preferred Cusine", "popular_dish_for_this_cuisine"]
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

encoded_train = encoder.fit_transform(train_df[categorical_cols])
encoded_train_df = pd.DataFrame(encoded_train, columns=encoder.get_feature_names_out(categorical_cols), index=train_df.index)
train_df = pd.concat([train_df.drop(columns=categorical_cols), encoded_train_df], axis=1)

encoded_test = encoder.transform(test_df[categorical_cols])
encoded_test_df = pd.DataFrame(encoded_test, columns=encoder.get_feature_names_out(categorical_cols), index=test_df.index)
test_df = pd.concat([test_df.drop(columns=categorical_cols), encoded_test_df], axis=1)

# Remove rows without target
train_df.dropna(subset=["dish"], inplace=True)
test_df.dropna(subset=["dish"], inplace=True)

label_encoder = LabelEncoder()
train_df["dish"] = label_encoder.fit_transform(train_df["dish"])
test_df["dish"] = label_encoder.transform(test_df["dish"])

# Set features + labels
X_train = train_df.drop(columns=["dish"])
y_train = train_df["dish"]
X_test = test_df.drop(columns=["dish"])
y_test = test_df["dish"]

# ----------------------------------------------------------
# TRAIN XGBOOST MODEL
# ----------------------------------------------------------

xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    learning_rate=0.05,
    max_depth=4,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=110
)

xgb_model.fit(X_train, y_train)
print("🚀 Model Training Completed")

# ----------------------------------------------------------
# EVALUATION
# ----------------------------------------------------------

y_pred = xgb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("🎯 Model Accuracy:", accuracy)

y_prob = xgb_model.predict_proba(X_test)
loss = log_loss(y_test, y_prob)
print("📉 Log Loss:", loss)

# Feature importance plot
xgb.plot_importance(xgb_model, max_num_features=12)
plt.show()


ModuleNotFoundError: No module named 'certifi'